In [5]:
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

# ------------------------------
# 1. Define the CNN Model
# ------------------------------
class CNN3LayerBN(nn.Module):
    def __init__(self):
        super(CNN3LayerBN, self).__init__()
        # Conv layers with batch norm
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        # Pooling and final linear layer
        self.pool = nn.MaxPool2d(2, 2)
        # After 3 pools of 2x2, 28x28 -> 14 -> 7 -> 3 (with padding=1, stride=1 conv, but pool reduces)
        # Actually: 28 -> after pool1:14, pool2:7, pool3:3 (if we pool after each conv)
        # So feature map size = 3x3, channels=128 -> 128*3*3 = 1152
        self.fc = nn.Linear(128 * 3 * 3, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # 28->14
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # 14->7
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # 7->3
        x = x.view(-1, 128 * 3 * 3)
        x = self.fc(x)
        return x

# ------------------------------
# 2. Training & Evaluation
# ------------------------------
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} '
                  f'({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')

def test(model, device, test_loader, criterion):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n')
    return accuracy


In [ ]:
class args:
    batch_size: int = 256
    test_batch_size: int = 1000
    epochs: int = 10
    lr: float = 0.001
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42
    save_model: bool = True


torch.manual_seed(args.seed)
device = torch.device(args.device)

# Data preparation (auto-download)
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),  # MNIST mean and std
    ]
)

train_dataset = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=args.test_batch_size, shuffle=False)

# Model, optimizer, loss
model = CNN3LayerBN().to(device)
optimizer = optim.Adam(model.parameters(), lr=args.lr)
criterion = nn.CrossEntropyLoss()

print(f"Using device: {device}")
print(model)

# Training loop
for epoch in range(1, args.epochs + 1):
    train(model, device, train_loader, optimizer, criterion, epoch)
    test(model, device, test_loader, criterion)

# Optional save
if args.save_model:
    torch.save(model.state_dict(), "mnist_cnn_bn.pt")
    print("Model saved to mnist_cnn_bn.pt")

Using device: cuda
CNN3LayerBN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=1152, out_features=10, bias=True)
)
Train Epoch: 1 [0/60000 (0%)]	Loss: 2.489490


Train Epoch: 1 [25600/60000 (43%)]	Loss: 0.075715
Train Epoch: 1 [51200/60000 (85%)]	Loss: 0.084153

Test set: Average loss: 0.0000, Accuracy: 9879/10000 (98.79%)

Train Epoch: 2 [0/60000 (0%)]	Loss: 0.045860
Train Epoch: 2 [25600/60000 (43%)]	Loss: 0.039044
Train Epoch: 2 [51200/60000 (85%)]	Loss: 0.035790

Test set: Average loss: 0.0000, Accuracy: 9867/10000 (98.67%)

Train Epoch: 3 [0/60000 (0%)]	Loss: 0.014984
Train Epoch: 3 [25600/60000 (43%)]	Loss: 0.007633
Train Epoch: 3 [51200/60000 (85%)]	Loss: 0.008692

Test set: Average loss: 0.0000, Accuracy: 9916/10000 (99.16%)

Train Epoch: 4 [0/60000 (0%)]	Loss: 0.015949
Train Epoch: 4 [25600/60000 (43%)]	Loss: 0.014085
Train Epoch: 4 [51200/60000 (85%)]	Loss: 0.029714

Test set: Average loss: 0.0000, Accuracy: 9921/10000 (99.21%)

Train Epoch: 5 [0/60000 (0%)]	Loss: 0.008052
Train Epoch: 5 [25600/60000 (43%)]	Loss: 0.003353
Train Epoch: 5 [51200/60000 (85%)]	Loss: 0.003800

Test set: Average loss: 0.0000, Accuracy: 9912/10000 (99.12%)

